# Feature Engineering — EV Charging Segment Dataset

Hybrid modelling pipeline:

| Question | Model | Target |
|---|---|---|
| *Where* to place EV stations? | Binary classifier | `y_clf = (nearest_station_m > 10 km)` |
| *How many* stations per optimal place? | Integer regressor | `y_reg = max(0, ⌈road_total_length_m / 30 km⌉ − road_n_stations)` |

### Notebook map

1. **Setup** — imports, load, normalise columns.
2. **Missing-value analysis** — business-driven imputation / row-drop decisions.
3. **Targets & leakage audit** — derived targets and the columns that mustn't leak.
4. **EV-domain features** (EV-expert hat) — AFIR, power, coverage, grid, scale.
5. **Statistical transforms** (data-scientist hat) — logs, bins, interactions, encoding.
6. **Collinearity audit** — correlation scan, VIF before / prune / VIF after.
7. **Assemble** — clean feature matrix + persist CSVs.
8. **Handoff summary** — what the modelling notebook inherits.

---
## 1 · Setup

### 1.1 Imports & display options

In [1]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

### 1.2 Load the segment dataset

In [2]:
DATA_PATH = "C:/Users/vldma/Datathon/Datasets/Modelling datasets/ev_charging_segment_ml_dataset.csv"
df = pd.read_csv(DATA_PATH)
print("loaded shape:", df.shape)
df.head()

loaded shape: (1535, 48)


,Carretera,Tipo_de_via,length_m,nearest_station_m,has_gap,segment_id,road_n_stations,road_total_refill_points,road_mean_power_kw,road_max_power_kw,road_n_operators,road_has_chademo,road_has_ccs,road_total_length_m,road_n_segments,road_n_gaps,road_gap_ratio,road_mean_nearest_m,road_max_nearest_m,road_centroid_lat,road_centroid_lon,road_min_lat,road_max_lat,road_min_lon,road_max_lon,road_span_lat,road_span_lon,nearest_grid_dist_deg,grid_capacity_occupied_mw,grid_capacity_pending_mw,max_voltage_kv,comunidad_autonoma,provincia,municipio,grid_capacity_3nearest_mw,grid_maxvoltage_3nearest_kv,ev_fleet_2027,ev_reg_2026,ev_reg_2027,ev_growth_rate,road_type_rank,road_stations_per_100km,road_refill_per_100km,is_underserved,is_desert,stations_deficit_afir,priority_score,priority_score_norm
0,TO-23,Multicarril,7527.979245,74.822269,0,0,12,45,83.450000,250.0,7,1,1,7527.979245,1.0,0.0,0.0,74.822269,74.822269,39.864278,-3.972001,39.862125,39.869240,-3.980628,-3.959182,0.007115,0.021446,0.508059,74.26,0.00,15.0,04 - Illes Balears,Illes Balears,Andratx,169.22,15.0,327883,76108,88118,1.157802,2,159.405328,597.769980,0,0,0,8.656784,0.210401
1,AP-7S,Autopista peaje,22078.044641,1902.155065,0,1,3,4,27.333333,30.0,2,1,1,90062.224503,5.0,0.0,0.0,3104.503662,4269.023990,36.546676,-4.760377,36.414670,36.613167,-5.225154,-4.527310,0.198497,0.697844,0.048716,46.33,0.01,66.0,01 - Andalucía,Málaga,Marbella,146.49,66.0,327883,76108,88118,1.157802,3,3.331030,4.441374,0,0,0,22.653805,0.550594
2,AP-7S,Autopista peaje,24146.013100,3027.787481,0,2,3,4,27.333333,30.0,2,1,1,90062.224503,5.0,0.0,0.0,3104.503662,4269.023990,36.546676,-4.760377,36.414670,36.613167,-5.225154,-4.527310,0.198497,0.697844,0.048716,46.33,0.01,66.0,01 - Andalucía,Málaga,Marbella,146.49,66.0,327883,76108,88118,1.157802,3,3.331030,4.441374,0,0,0,24.047753,0.584473
3,AP-7S,Autopista peaje,28323.852697,4269.023990,0,3,3,4,27.333333,30.0,2,1,1,90062.224503,5.0,0.0,0.0,3104.503662,4269.023990,36.546676,-4.760377,36.414670,36.613167,-5.225154,-4.527310,0.198497,0.697844,0.048716,46.33,0.01,66.0,01 - Andalucía,Málaga,Marbella,146.49,66.0,327883,76108,88118,1.157802,3,3.331030,4.441374,0,0,0,25.078124,0.609516
4,AP-7S,Autopista peaje,11492.490960,4014.936194,0,4,3,4,27.333333,30.0,2,1,1,90062.224503,5.0,0.0,0.0,3104.503662,4269.023990,36.546676,-4.760377,36.414670,36.613167,-5.225154,-4.527310,0.198497,0.697844,0.048716,46.33,0.01,66.0,01 - Andalucía,Málaga,Marbella,146.49,66.0,327883,76108,88118,1.157802,3,3.331030,4.441374,0,0,0,24.894077,0.605043


### 1.3 Normalise column names

Lowercase + snake_case so every downstream reference is consistent.

In [3]:
df.columns = [c.lower().strip().replace(" ", "_") for c in df.columns]
df.dtypes

carretera                       object
tipo_de_via                     object
length_m                       float64
nearest_station_m              float64
has_gap                          int64
segment_id                       int64
road_n_stations                  int64
road_total_refill_points         int64
road_mean_power_kw             float64
road_max_power_kw              float64
road_n_operators                 int64
road_has_chademo                 int64
road_has_ccs                     int64
road_total_length_m            float64
road_n_segments                float64
road_n_gaps                    float64
road_gap_ratio                 float64
road_mean_nearest_m            float64
road_max_nearest_m             float64
road_centroid_lat              float64
road_centroid_lon              float64
road_min_lat                   float64
road_max_lat                   float64
road_min_lon                   float64
road_max_lon                   float64
road_span_lat            

---
## 2 · Missing-value analysis

Each bucket of missing values gets a different treatment, justified by the business meaning of the gap.

| Bucket | Columns | % missing | Meaning | Action |
|---|---|---|---|---|
| **Grid** | `nearest_grid_dist_deg`, `grid_capacity_*`, `max_voltage_kv`, `grid_*_3nearest_*` | ~16.7 % | Segment outside published Endesa grid coverage — *informative missingness* | Flag `grid_data_missing`; per-region median imputation |
| **Road-geometry** | `road_centroid_*`, `road_min/max_lat/lon`, `road_span_*` | ~16 % | Road's geographic extent could not be computed | Flag `road_geo_missing`; per-region mean imputation |
| **Orphan rows** | `tipo_de_via` missing (11 rows) | 0.7 % | Unclassified segment; no road context recoverable | **Drop** the rows |
| **Admin** | `comunidad_autonoma`, `provincia`, `municipio` | residual | Missing geocoding string | Fill with `"unknown"` |

### 2.1 Missingness scan

In [4]:
missing = df.isna().sum()
missing_summary = (
    pd.DataFrame({
        "n_missing": missing,
        "pct_missing": (missing / len(df) * 100).round(2),
    })
    .query("n_missing > 0")
    .sort_values("n_missing", ascending=False)
)
missing_summary

,n_missing,pct_missing
road_centroid_lat,256,16.68
road_span_lon,256,16.68
road_span_lat,256,16.68
road_max_lon,256,16.68
road_min_lon,256,16.68
road_max_lat,256,16.68
road_min_lat,256,16.68
road_centroid_lon,256,16.68
grid_capacity_occupied_mw,256,16.68
grid_capacity_pending_mw,256,16.68


### 2.2 Grid block — flag & impute

The grid columns come from Endesa's published capacity map; absence often means the segment lies outside that operator's footprint. The missingness itself carries signal, so we preserve it via `grid_data_missing` and fill the numerics with per-region medians (global median as fallback).

In [5]:
grid_cols = [
    "nearest_grid_dist_deg",
    "grid_capacity_occupied_mw",
    "grid_capacity_pending_mw",
    "max_voltage_kv",
    "grid_capacity_3nearest_mw",
    "grid_maxvoltage_3nearest_kv",
]
grid_cols = [c for c in grid_cols if c in df.columns]

df["grid_data_missing"] = df[grid_cols].isna().any(axis=1).astype(int)
print("grid_data_missing counts:", df["grid_data_missing"].value_counts().to_dict())

for col in grid_cols:
    regional_median = df.groupby("comunidad_autonoma")[col].transform("median")
    df[col] = df[col].fillna(regional_median).fillna(df[col].median())

print("grid NaNs after imputation:", df[grid_cols].isna().sum().sum())

grid_data_missing counts: {0: 1279, 1: 256}
grid NaNs after imputation: 0


### 2.3 Road-geometry block — flag & impute

Centroid & bounding-box columns are missing whenever the upstream shapefile couldn't compute a road's geographic extent. We flag and use per-region means (less sensitive than medians for coordinate data).

In [6]:
road_geo_cols = [
    "road_centroid_lat", "road_centroid_lon",
    "road_min_lat", "road_max_lat", "road_min_lon", "road_max_lon",
    "road_span_lat", "road_span_lon",
]
road_geo_cols = [c for c in road_geo_cols if c in df.columns]

df["road_geo_missing"] = df[road_geo_cols].isna().any(axis=1).astype(int)
print("road_geo_missing counts:", df["road_geo_missing"].value_counts().to_dict())

for col in road_geo_cols:
    regional_mean = df.groupby("comunidad_autonoma")[col].transform("mean")
    df[col] = df[col].fillna(regional_mean).fillna(df[col].mean())

road_geo_missing counts: {0: 1279, 1: 256}


### 2.4 Drop orphan rows, fill admin strings, assert clean

11 rows have `tipo_de_via` missing and no recoverable road context — drop them. Any residual admin-string gaps are filled with `"unknown"`. Final assertion: the dataframe has zero NaNs before we move on.

In [7]:
before = len(df)
df = df.dropna(subset=["tipo_de_via"]).reset_index(drop=True)
print(f"dropped {before - len(df)} orphan-road rows; remaining: {len(df)}")

for col in ["comunidad_autonoma", "provincia", "municipio"]:
    if col in df.columns:
        df[col] = df[col].fillna("unknown")

remaining = df.isna().sum()
remaining = remaining[remaining > 0]
print("remaining NaN columns:", dict(remaining) if len(remaining) else "none")
assert df.isna().sum().sum() == 0, "unhandled NaNs remain"

dropped 11 orphan-road rows; remaining: 1524
remaining NaN columns: none


---
## 3 · Targets & leakage audit

### 3.0 Why the pre-baked targets don't work

The shipped labels collapse on this data:
- `is_underserved`: **2 / 1 535** positives (0.13 %) — unusable for supervised classification.
- `stations_deficit_afir`: **all zeros** — no segment breaches the 60 km AFIR rule.
- `is_desert` and `has_gap` collapse the same way.

**Root cause.** The 60 km AFIR threshold is simply not breached on the Spanish interurban network today — current coverage already exceeds that regulatory minimum.

### 3.0.1 Pivot to business-driven derived targets

| Target | Definition | Rationale | Distribution |
|---|---|---|---|
| **`y_clf`** — *where?* | `(nearest_station_m > 10_000).astype(int)` | 10 km sits below AFIR but above trivial distances — matches real EV driver expectations | ~16 % positive |
| **`y_reg`** — *how many?* | `max(0, ⌈road_total_length_m / 30_000⌉ − road_n_stations)` | Deficit to a 1-per-30 km operational rule (industry planning heuristic, tighter than AFIR) | 75 % zeros, max 11, std ≈ 1.1 |

### 3.0.2 Leakage discipline

Any column that directly composes either target — or is an encoded form of such a column — goes into `LEAKAGE_COLS` and is stripped from the feature matrix in §7.

### 3.1 Build derived targets

In [8]:
# Binary: is the segment > 10 km from a charger?
y_clf = (df["nearest_station_m"] > 10_000).astype(int).copy()

# Integer: extra stations needed on this road to hit 1/30 km density
y_reg = np.maximum(
    0,
    np.ceil(df["road_total_length_m"] / 30_000) - df["road_n_stations"],
).astype(int).copy()

print("--- legacy (degenerate) targets ---")
print("is_underserved counts:", df["is_underserved"].value_counts().to_dict())
print("stations_deficit_afir describe:\n", df["stations_deficit_afir"].describe())

print("\n--- derived y_clf (>10 km proxy) ---")
print(y_clf.value_counts(normalize=True).round(3))

print("\n--- derived y_reg (1/30 km deficit) ---")
print(y_reg.describe())

--- legacy (degenerate) targets ---
is_underserved counts: {0: 1522, 1: 2}
stations_deficit_afir describe:
 count    1524.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: stations_deficit_afir, dtype: float64

--- derived y_clf (>10 km proxy) ---
nearest_station_m
0    0.837
1    0.163
Name: proportion, dtype: float64

--- derived y_reg (1/30 km deficit) ---
count    1524.000000
mean        0.423885
std         1.065230
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        11.000000
dtype: float64


### 3.2 Declare the leakage set

`LEAKAGE_COLS` lists exact column names; `LEAKAGE_PREFIXES` catches OHE-expansions of leaking categoricals (e.g. `nearest_station_band_*`). `is_leakage()` is the single predicate used everywhere downstream.

In [9]:
LEAKAGE_COLS = {
    # Legacy targets and derived ranking columns
    "is_underserved", "is_desert", "has_gap",
    "stations_deficit_afir", "priority_score", "priority_score_norm",
    # y_clf direct inputs / encoded forms
    "nearest_station_m", "log_nearest_station_m",
    "afir_distance_ratio", "afir_distance_breach",
    # y_reg direct inputs / encoded forms
    "road_total_length_m", "log_road_total_length_m",
    "road_n_stations",
    "road_stations_per_100km", "log_road_stations_per_100km",
    "segment_share_of_road",
}
LEAKAGE_PREFIXES = ("nearest_station_band_",)

def is_leakage(col: str) -> bool:
    return col in LEAKAGE_COLS or col.startswith(LEAKAGE_PREFIXES)

print(f"LEAKAGE_COLS: {len(LEAKAGE_COLS)} explicit names + prefixes {LEAKAGE_PREFIXES}")

LEAKAGE_COLS: 16 explicit names + prefixes ('nearest_station_band_',)


---
## 4 · EV-domain features (EV-expert hat)

Reference: **EU Regulation 2023/1804 (AFIR)** — TEN-T core roads must have fast chargers (≥150 kW) every ≤60 km. These rules are the yardstick for "is this segment adequately served?".

### 4.1 AFIR compliance signals

In [10]:
df["afir_distance_ratio"] = df["nearest_station_m"] / 60_000
df["afir_distance_breach"] = (df["nearest_station_m"] > 60_000).astype(int)
df["afir_power_gap_kw"] = np.clip(150 - df["road_max_power_kw"], 0, None)
df["afir_power_compliant"] = (df["road_max_power_kw"] >= 150).astype(int)

df[["afir_distance_ratio", "afir_distance_breach",
    "afir_power_gap_kw", "afir_power_compliant"]].describe()

,afir_distance_ratio,afir_distance_breach,afir_power_gap_kw,afir_power_compliant
count,1524.000000,1524.0,1524.000000,1524.000000
mean,0.086354,0.0,49.947060,0.534121
std,0.107642,0.0,60.544965,0.498998
min,0.000664,0.0,0.000000,0.000000
25%,0.016371,0.0,0.000000,0.000000
50%,0.046582,0.0,0.000000,1.000000
75%,0.109814,0.0,100.000000,1.000000
max,0.865120,0.0,150.000000,1.000000


### 4.2 Charging-power mix

In [11]:
df["power_range_kw"] = df["road_max_power_kw"] - df["road_mean_power_kw"]
df["has_hpc"] = (df["road_max_power_kw"] >= 250).astype(int)
df["connector_diversity"] = df["road_has_chademo"].astype(int) + df["road_has_ccs"].astype(int)

df[["power_range_kw", "has_hpc", "connector_diversity"]].describe()

,power_range_kw,has_hpc,connector_diversity
count,1524.000000,1524.000000,1524.000000
mean,113.506764,0.353675,1.508530
std,116.774200,0.478267,0.840102
min,0.000000,0.000000,0.000000
25%,4.866667,0.000000,1.000000
50%,80.669286,0.000000,2.000000
75%,232.294074,1.000000,2.000000
max,468.200795,1.000000,2.000000


### 4.3 Coverage & site intensity

In [12]:
df["refill_per_station"] = df["road_total_refill_points"] / df["road_n_stations"].clip(lower=1)
df["coverage_ratio"] = 1 - df["road_gap_ratio"]
df["gap_severity"] = df["road_max_nearest_m"] / df["road_mean_nearest_m"].clip(lower=1)

df[["refill_per_station", "coverage_ratio", "gap_severity"]].describe()

,refill_per_station,coverage_ratio,gap_severity
count,1524.000000,1524.000000,1524.000000
mean,2.629336,0.998688,2.300111
std,1.875965,0.017634,1.143080
min,0.000000,0.666667,1.000000
25%,1.750000,1.000000,1.381276
50%,2.741935,1.000000,2.132682
75%,3.422414,1.000000,3.041747
max,26.000000,1.000000,6.074257


### 4.4 Grid-readiness

Headroom = aggregated capacity across the three nearest substations minus the load already occupied at the nearest one. `grid_strong_hv` flags ≥132 kV (transmission-level) connections.

In [13]:
df["grid_headroom_mw"] = df["grid_capacity_3nearest_mw"] - df["grid_capacity_occupied_mw"]
df["grid_strong_hv"] = (df["max_voltage_kv"] >= 132).astype(int)
df["grid_capacity_per_station"] = df["grid_capacity_3nearest_mw"] / df["road_n_stations"].clip(lower=1)

df[["grid_headroom_mw", "grid_strong_hv", "grid_capacity_per_station"]].describe()

,grid_headroom_mw,grid_strong_hv,grid_capacity_per_station
count,1524.000000,1524.000000,1524.000000
mean,47.426831,0.314304,19.784701
std,49.878054,0.464391,32.492531
min,0.000000,0.000000,0.000000
25%,10.790000,0.000000,1.162857
50%,37.810000,0.000000,5.650741
75%,59.100000,1.000000,41.395000
max,352.710000,1.000000,412.760000


### 4.5 Segment scale

In [14]:
df["segment_share_of_road"] = df["length_m"] / df["road_total_length_m"].clip(lower=1)
df[["length_m", "road_total_length_m", "segment_share_of_road"]].describe()

,length_m,road_total_length_m,segment_share_of_road
count,1524.000000,1524.000000,1524.000000
mean,18889.942068,170293.655370,0.324147
std,30119.977626,192789.185914,0.371365
min,0.103228,211.109602,0.000001
25%,1798.936483,12230.258632,0.033649
50%,5487.952210,78793.657367,0.138246
75%,21188.822578,351678.182086,0.555365
max,232827.166963,689343.442490,1.000000


### 4.6 Drop Spain-level constants

`ev_fleet_2027`, `ev_reg_2026`, `ev_reg_2027`, `ev_growth_rate` are country-level forecasts — identical across every row, so they add no signal.

In [15]:
demand_cols = ["ev_fleet_2027", "ev_reg_2026", "ev_reg_2027", "ev_growth_rate"]
demand_cols = [c for c in demand_cols if c in df.columns]
unique_counts = {c: df[c].nunique() for c in demand_cols}
print("unique values per demand column:", unique_counts)

constants_to_drop = [c for c, n in unique_counts.items() if n <= 1]
print("dropping constants:", constants_to_drop)
df = df.drop(columns=constants_to_drop)

unique values per demand column: {'ev_fleet_2027': 1, 'ev_reg_2026': 1, 'ev_reg_2027': 1, 'ev_growth_rate': 1}
dropping constants: ['ev_fleet_2027', 'ev_reg_2026', 'ev_reg_2027', 'ev_growth_rate']


---
## 5 · Statistical transforms (data-scientist hat)

### 5.1 Log-transform skewed distance / density columns

Linear models benefit from variance-stabilising distance and density features. `log1p` handles the zeros safely.

In [16]:
log_cols = [
    "length_m",
    "nearest_station_m",
    "road_total_length_m",
    "road_max_nearest_m",
    "road_mean_nearest_m",
    "road_stations_per_100km",
    "road_refill_per_100km",
]
for col in log_cols:
    if col in df.columns:
        df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))

df[[f"log_{c}" for c in log_cols if c in df.columns]].describe()

,log_length_m,log_nearest_station_m,log_road_total_length_m,log_road_max_nearest_m,log_road_mean_nearest_m,log_road_stations_per_100km,log_road_refill_per_100km
count,1524.000000,1524.000000,1524.000000,1524.000000,1524.000000,1524.000000,1524.000000
mean,8.562297,7.818241,10.877972,8.875092,8.161169,2.310605,3.141383
std,1.952865,1.325354,1.922491,1.269340,1.008769,1.452668,1.796950
min,0.098240,3.709159,5.357103,4.328392,4.328392,0.000000,0.000000
25%,7.495506,6.890873,9.411750,8.108148,7.760554,1.465805,2.397529
50%,8.610492,7.935916,11.274600,9.329747,8.294166,2.374504,3.337762
75%,9.961273,8.793286,12.770332,9.817743,8.772197,3.147660,4.330496
max,12.358056,10.857232,13.443496,10.857232,10.360625,6.389513,8.022291


### 5.2 AFIR-aligned bins + hand-picked interactions

Bins (`nearest_station_band`) help linear models; trees ignore them. Two interactions only — no polynomial fan-out.

In [17]:
df["nearest_station_band"] = pd.cut(
    df["nearest_station_m"],
    bins=[-np.inf, 5_000, 20_000, 60_000, np.inf],
    labels=["well_served", "moderate", "stretched", "breach"],
)

df["demand_vs_supply"] = df["log_length_m"] * (1 / (df["road_stations_per_100km"] + 1))
df["grid_x_power"] = df["grid_headroom_mw"] * df["road_max_power_kw"]

df[["nearest_station_band", "demand_vs_supply", "grid_x_power"]].head()

,nearest_station_band,demand_vs_supply,grid_x_power
0,well_served,0.055650,23740.0
1,well_served,2.309470,3004.8
2,well_served,2.330142,3004.8
3,well_served,2.366988,3004.8
4,well_served,2.158733,3004.8


### 5.3 Drop identifiers & one-hot encode

Drop `carretera`, `provincia`, `municipio`, `segment_id` (identifier-like / high cardinality). One-hot the three low-cardinality categoricals with `drop_first=True` to avoid the dummy-variable trap.

In [18]:
drop_as_id = [c for c in ["carretera", "provincia", "municipio", "segment_id"] if c in df.columns]
df = df.drop(columns=drop_as_id)
print("dropped identifiers:", drop_as_id)

ohe_cols = [c for c in ["tipo_de_via", "comunidad_autonoma", "nearest_station_band"] if c in df.columns]
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True, dtype=int)
print("one-hot expanded:", ohe_cols)
print("shape after encoding:", df.shape)

dropped identifiers: ['carretera', 'provincia', 'municipio', 'segment_id']
one-hot expanded: ['tipo_de_via', 'comunidad_autonoma', 'nearest_station_band']
shape after encoding: (1524, 79)


---
## 6 · Collinearity audit

1. **6.1 Correlation scan** — list the top absolute correlations and the pairs above the 0.95 cut.
2. **6.2 VIF *before* prune** — per-feature Variance Inflation Factor = `1 / (1 − R²)`; rule of thumb **VIF > 10 = severe** multicollinearity, 5–10 = moderate.
3. **6.3 Prune** — for every `|r|>0.95` pair drop the member with the higher VIF (logged per drop).
4. **6.4 VIF *after* prune** — sanity-check that multicollinearity has actually been reduced.

### 6.1 Correlation scan

In [19]:
numeric_pool = (
    df[[c for c in df.columns if not is_leakage(c)]]
    .select_dtypes(include=[np.number])
    .columns.tolist()
)
print(f"numeric pool after leakage filter: {len(numeric_pool)} features")

corr = df[numeric_pool].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs_all = upper.stack().dropna().sort_values(ascending=False)
high_pairs = pairs_all.loc[pairs_all > 0.95]

print("\ntop-15 absolute correlations:")
print(pairs_all.head(15))
print(f"\npairs with |r| > 0.95 — {len(high_pairs)} flagged:")
print(high_pairs)

numeric pool after leakage filter: 60 features

top-15 absolute correlations:
grid_data_missing          comunidad_autonoma_unknown    1.000000
                           road_geo_missing              1.000000
road_geo_missing           comunidad_autonoma_unknown    1.000000
road_gap_ratio             coverage_ratio                1.000000
road_has_chademo           connector_diversity           0.977667
road_has_ccs               connector_diversity           0.975307
road_max_power_kw          power_range_kw                0.970511
road_centroid_lon          road_max_lon                  0.961323
road_centroid_lat          road_max_lat                  0.952855
log_road_max_nearest_m     log_road_mean_nearest_m       0.932742
grid_capacity_3nearest_mw  grid_headroom_mw              0.931569
road_centroid_lon          road_min_lon                  0.917647
road_n_gaps                coverage_ratio                0.916031
                           road_gap_ratio                0.91603

### 6.2 VIF — *before* prune

In [20]:
def vif_scores(X: pd.DataFrame) -> pd.Series:
    """Per-column VIF = 1 / (1 - R²) via OLS in pure numpy. Drops constants."""
    X = X.astype(float).copy()
    X = X.loc[:, X.nunique() > 1]
    Xc = X - X.mean(axis=0)
    out = {}
    for col in X.columns:
        y = Xc[col].to_numpy()
        rest = Xc.drop(columns=col).to_numpy()
        beta, *_ = np.linalg.lstsq(rest, y, rcond=None)
        y_hat = rest @ beta
        ssr = float(np.sum((y - y_hat) ** 2))
        sst = float(np.sum(y ** 2))
        r2 = 1.0 - ssr / sst if sst > 0 else 0.0
        out[col] = np.inf if r2 >= 1 - 1e-10 else 1.0 / (1.0 - r2)
    return pd.Series(out).sort_values(ascending=False)

vif_before = vif_scores(df[numeric_pool])
print(f"VIF summary BEFORE prune — {len(vif_before)} features")
print(f"  severe   (VIF > 10): {(vif_before > 10).sum()}")
print(f"  moderate (5 < VIF ≤ 10): {((vif_before > 5) & (vif_before <= 10)).sum()}")
print(f"  ok       (VIF ≤ 5): {(vif_before <= 5).sum()}")
print("\ntop-20 VIF:")
vif_before.head(20).to_frame("vif")

VIF summary BEFORE prune — 60 features
  severe   (VIF > 10): 43
  moderate (5 < VIF ≤ 10): 7
  ok       (VIF ≤ 5): 10

top-20 VIF:


,vif
road_max_power_kw,inf
road_mean_power_kw,inf
road_has_ccs,inf
road_has_chademo,inf
road_gap_ratio,inf
grid_capacity_3nearest_mw,inf
road_span_lon,inf
road_span_lat,inf
road_min_lon,inf
road_max_lon,inf


### 6.3 Prune collinear pairs (VIF tie-break)

In [21]:
dropped_by_corr: set = set()
for idx, r in zip(high_pairs.index, high_pairs.values):
    a, b = idx
    if a in dropped_by_corr or b in dropped_by_corr:
        continue
    active = [c for c in numeric_pool if c not in dropped_by_corr]
    v = vif_scores(df[active])
    loser = a if v.get(a, 0) >= v.get(b, 0) else b
    print(f"  |r|={r:.3f}: {a} vs {b}  →  drop {loser} (VIF={v.get(loser, float('nan')):.2f})")
    dropped_by_corr.add(loser)

print(f"\ntotal dropped: {len(dropped_by_corr)}")
df = df.drop(columns=list(dropped_by_corr))
print("shape after prune:", df.shape)

  |r|=1.000: grid_data_missing vs comunidad_autonoma_unknown  →  drop grid_data_missing (VIF=inf)
  |r|=1.000: road_geo_missing vs comunidad_autonoma_unknown  →  drop road_geo_missing (VIF=inf)
  |r|=1.000: road_gap_ratio vs coverage_ratio  →  drop road_gap_ratio (VIF=inf)
  |r|=0.978: road_has_chademo vs connector_diversity  →  drop road_has_chademo (VIF=inf)
  |r|=0.975: road_has_ccs vs connector_diversity  →  drop road_has_ccs (VIF=33.62)
  |r|=0.971: road_max_power_kw vs power_range_kw  →  drop road_max_power_kw (VIF=inf)
  |r|=0.961: road_centroid_lon vs road_max_lon  →  drop road_max_lon (VIF=inf)
  |r|=0.953: road_centroid_lat vs road_max_lat  →  drop road_max_lat (VIF=inf)

total dropped: 8
shape after prune: (1524, 71)


### 6.4 VIF — *after* prune

In [22]:
numeric_pool_post = (
    df[[c for c in df.columns if not is_leakage(c)]]
    .select_dtypes(include=[np.number])
    .columns.tolist()
)
vif_after = vif_scores(df[numeric_pool_post])
print(f"VIF summary AFTER prune — {len(vif_after)} features")
print(f"  severe   (VIF > 10): {(vif_after > 10).sum()}")
print(f"  moderate (5 < VIF ≤ 10): {((vif_after > 5) & (vif_after <= 10)).sum()}")
print(f"  ok       (VIF ≤ 5): {(vif_after <= 5).sum()}")
print("\ntop-20 VIF:")
vif_after.head(20).to_frame("vif")

VIF summary AFTER prune — 52 features
  severe   (VIF > 10): 31
  moderate (5 < VIF ≤ 10): 11
  ok       (VIF ≤ 5): 10

top-20 VIF:


,vif
grid_capacity_occupied_mw,inf
grid_capacity_3nearest_mw,inf
grid_headroom_mw,inf
road_type_rank,inf
tipo_de_via_Autopista peaje,inf
tipo_de_via_Carretera convencional,inf
tipo_de_via_Multicarril,inf
log_road_max_nearest_m,184.054381
road_min_lat,166.805190
road_min_lon,155.405948


---
## 7 · Assemble & persist

Both `X_clf` and `X_reg` share the same feature matrix — only the target differs. The assertion in 7.1 guarantees no leakage column has slipped through.

### 7.1 Build the feature matrix

In [23]:
feature_cols = [c for c in df.columns if not is_leakage(c)]
X = df[feature_cols].copy()

y_clf = y_clf.loc[X.index]
y_reg = y_reg.loc[X.index]

assert not any(is_leakage(c) for c in X.columns), "leakage column slipped into X"
assert X.isna().sum().sum() == 0, "NaNs in X"

print("X.shape:", X.shape)
print("\ny_clf distribution:")
print(y_clf.value_counts(normalize=True).round(3))
print("\ny_reg distribution:")
print(y_reg.describe())

X.shape: (1524, 52)

y_clf distribution:
nearest_station_m
0    0.837
1    0.163
Name: proportion, dtype: float64

y_reg distribution:
count    1524.000000
mean        0.423885
std         1.065230
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        11.000000
dtype: float64


### 7.2 Write classification & regression CSVs

In [24]:
OUT_DIR = "C:/Users/vldma/Datathon/Datasets/Modelling datasets"
X_clf = X.assign(is_underserved=y_clf)
X_reg = X.assign(stations_deficit_afir=y_reg)
X_clf.to_csv(f"{OUT_DIR}/fe_features_classification.csv", index=False)
X_reg.to_csv(f"{OUT_DIR}/fe_features_regression.csv", index=False)

print("wrote:")
print(f"  {OUT_DIR}/fe_features_classification.csv  — {X_clf.shape}")
print(f"  {OUT_DIR}/fe_features_regression.csv      — {X_reg.shape}")

wrote:
  C:/Users/vldma/Datathon/Datasets/Modelling datasets/fe_features_classification.csv  — (1524, 53)
  C:/Users/vldma/Datathon/Datasets/Modelling datasets/fe_features_regression.csv      — (1524, 53)


---
## 8 · Handoff summary

| Step | In | Out |
|---|---|---|
| Rows | 1 535 | **1 524** (11 orphan-road rows dropped) |
| Features | 48 raw | **~52** after EV-domain + stats FE, minus leakage + constants + collinearity prune |

### Targets (derived — legacy targets are degenerate)
- `y_clf = (nearest_station_m > 10 km)` — binary *where-to-place* label, ~16 % positive.
- `y_reg = max(0, ⌈road_total_length_m / 30 km⌉ − road_n_stations)` — integer *how-many-to-add*, ~25 % non-zero, max 11.

### Leakage stripped
Pre-baked targets (`is_underserved`, `is_desert`, `has_gap`, `stations_deficit_afir`, `priority_score*`) plus every direct input and encoded form of the two derived targets. See `LEAKAGE_COLS` in §3.2.

### Imputation policy
- Grid block (~16.7 %): `grid_data_missing` flag + per-region median.
- Road-geometry block (~16 %): `road_geo_missing` flag + per-region mean.
- Orphan rows (0.7 %): dropped.
- Admin strings: filled with `"unknown"`.

### Files written
- `fe_features_classification.csv`
- `fe_features_regression.csv`

### Caveats for the modelling notebook
- The collinearity prune is greedy — it may drop a missingness flag if it is perfectly co-linear with `comunidad_autonoma_unknown`. That is redundancy, not a bug.
- `y_reg` is zero-inflated. A two-stage model (classifier for `deficit > 0`, then regressor on positives) typically beats a single regressor.